# Built-in `frozendict` (Python 3.15)

[PEP 814](https://peps.python.org/pep-0814/) adds `frozendict` as a built-in: a dict that can't be changed after it's made. Because it can't change, it can be **hashed**, so it works as a dict key, a set member or an `lru_cache` argument. A plain `dict` can't be any of those.


In [1]:
import sys

sys.version

'3.15.0b1 (main, May 10 2026, 19:28:01) [Clang 22.1.3 ]'

## Create it like a dict, no import needed


In [2]:
config = frozendict(gamma=0.99, lr=1e-3)
print(config)
print(frozendict({"a": 1}), frozendict([("a", 1)]), frozendict.fromkeys("xy", 0))

frozendict({'gamma': 0.99, 'lr': 0.001})
frozendict({'a': 1}) frozendict({'a': 1}) frozendict({'x': 0, 'y': 0})


## Reads work like a dict


In [3]:
print(config["gamma"], config.get("missing", "default"), "lr" in config, len(config))
print(list(config.keys()), list(config.items()))

0.99 default True 2
['gamma', 'lr'] [('gamma', 0.99), ('lr', 0.001)]


## Writes are blocked

There's no `__setitem__` or `__delitem__` at all, and no `update`, `pop` or `clear`.


In [4]:
try:
    config["gamma"] = 0.5
except TypeError as e:
    print("TypeError:", e)
try:
    del config["gamma"]
except TypeError as e:
    print("TypeError:", e)
print("has update/pop/clear?", [hasattr(config, m) for m in ("update", "pop", "clear")])

TypeError: 'frozendict' object does not support item assignment
TypeError: 'frozendict' object does not support item deletion
has update/pop/clear? [False, False, False]


## It's a copy, not a view

Unlike `types.MappingProxyType`, changing the source dict afterwards doesn't show through.


In [5]:
source = {"gamma": 0.99}
fd = frozendict(source)
source["gamma"] = 0.5
print("source changed to", source["gamma"], "but frozendict still has", fd["gamma"])

source changed to 0.5 but frozendict still has 0.99


## Hashable: usable as a dict key or set member

Key order doesn't matter for `==` or `hash`, same as `dict ==`.


In [6]:
a = frozendict(x=1, y=2)
b = frozendict(y=2, x=1)
print("a == b:", a == b, "| hash equal:", hash(a) == hash(b))
results = {a: "run 1"}
print("lookup with b:", results[b])
print("set dedupes:", {a, b})
print("equal to a plain dict:", a == {"x": 1, "y": 2})

a == b: True | hash equal: True
lookup with b: run 1
set dedupes: {frozendict({'x': 1, 'y': 2})}
equal to a plain dict: True


## Values must be hashable for `hash()` to work


In [7]:
try:
    hash(frozendict(layers=[64, 64]))
except TypeError as e:
    print("TypeError:", e)
print("use a tuple instead:", hash(frozendict(layers=(64, 64))) is not None)

TypeError: unhashable type: 'list'
use a tuple instead: True


## Works as an `lru_cache` argument (a plain dict can't)

The same config in a different order is a cache hit.


In [8]:
from functools import lru_cache


@lru_cache
def train(cfg):
    print("  training with", dict(cfg))
    return cfg["lr"] * 1000


train(frozendict(lr=1e-3, gamma=0.99))
train(frozendict(gamma=0.99, lr=1e-3))
print(train.cache_info())
try:
    train({"lr": 1e-3})
except TypeError as e:
    print("plain dict -> TypeError:", e)

  training with {'lr': 0.001, 'gamma': 0.99}
CacheInfo(hits=1, misses=1, maxsize=128, currsize=1)
plain dict -> TypeError: unhashable type: 'dict'


## `|` returns a new frozendict and leaves the original alone


In [9]:
bigger = config | {"batch_size": 64}
print(bigger, type(bigger).__name__)
print("original unchanged:", config)

frozendict({'gamma': 0.99, 'lr': 0.001, 'batch_size': 64}) frozendict
original unchanged: frozendict({'gamma': 0.99, 'lr': 0.001})


## Type relationships

It's a `Mapping` but **not** a `dict` subclass, so `isinstance(x, dict)` checks will reject it.


In [10]:
import json
from collections.abc import Mapping, MutableMapping

print("isinstance dict:", isinstance(config, dict))
print("isinstance Mapping:", isinstance(config, Mapping))
print("isinstance MutableMapping:", isinstance(config, MutableMapping))
print("json:", json.dumps(config))

isinstance dict: False
isinstance Mapping: True
isinstance MutableMapping: False
json: {"gamma": 0.99, "lr": 0.001}


In [1]:
user = frozendict(name='rahul', age=27)
new_user = user | dict(name='mehak')
print(new_user)

frozendict({'name': 'mehak', 'age': 27})


In [2]:
user = frozendict(name='rahul', age=27)




name = user['name']
age = user['age']
print(name)
print(age)

rahul
27
